# ANN backend — Fingerprint Embedding Network (9, 33, 384)

Speaker verification on **triplet-STDP** weight fingerprints produced by
`prepare_fingerprints.ipynb`. Each sample is stored **compact** and expanded to the model
input `(9, 33, 384)` **at model time**:

| channels | source | meaning |
|---|---|---|
| `0:4` | `weights (4,33,384)` | STDP weight type-images (4 input-neuron types) |
| `4:8` | `input_activity (384,)` gathered via `IN_IDX` | presynaptic input firing rate, co-registered |
| `8:9` | `hidden_activity (384,)` broadcast over offset | hidden firing rate |

**Train / eval split (open-set):**
- **dev / train** = VoxCeleb2 shards → `vox2_*_fingerprints.npz` (closed-set classification target).
- **test / eval**  = VoxCeleb1 shards → `vox1_*_fingerprints.npz`, evaluated open-set.
  VoxCeleb2 (`id0xxxx`) and VoxCeleb1 (`id1xxxx`) speaker sets are disjoint.

---

## What changed vs. the OOM version

**1. Zero-copy shard access (`np.memmap` into the uncompressed `.npz`).**
The old notebook held one ~10 GB dev shard-group *plus* the entire 4.18 GB test split in
anonymous RAM, ~18 GB of a 29 GB box. Group packing was re-randomised each epoch, so peak
RSS varied epoch to epoch and eventually tipped over (epoch 7).

Now nothing is resident. Every shard member is memory-mapped at its byte offset inside the
zip container; hot rows live in the **page cache**, which the kernel reclaims under pressure
instead of OOM-killing. Anonymous RSS during training is a few hundred MB.

**2. The whole shard-group streaming machinery is deleted.**
This is not just a memory win. Under group streaming, a PK batch could only draw its 32
classes from the ~2000 speakers resident in the current group — the AAM head saw each of the
6112 classes in bursts of one third of an epoch. Now PK batches sample the full class set.

**3. Session-leakage-free evaluation.**
With 2 fingerprints per session, every test sample has *exactly one same-session positive*,
and cosine will find it trivially — same mic, same channel, same ambient noise. The old
`eval_metrics` masked self-matches but never touched `record_ids`. R@1/EER were therefore
part speaker verification, part **session re-identification**.

`eval_metrics` now excludes same-recording pairs from Rank-k, mAP *and* the EER histograms.
Both numbers are reported each epoch (`R@1` / `R@1*`) so the gap is visible — that gap is
itself a reportable result. **Model selection uses the session-free EER.**

**4. Crash-safe checkpointing.**
Every epoch writes model + AAM + optimizer + scheduler + scaler + history to `last_*.pth`.
Set `RESUME = True` to pick up where a dead kernel left off.

### Ablation selector — `FINGERPRINT_MODE`
- **`"full"`** — all 9 channels.  **`"rates_only"`** — weights neutralised; rates kept.
- **`"weights_only"`** — rates neutralised; weights kept.
- **`"input_rate_only"` / `"hidden_rate_only"`** — keep only one rate group.

Ablated channels are replaced by a zero-variance constant (`exp(-30)`, neutralised by the
input BatchNorm), so architecture and param count are **identical across modes**.


In [ ]:
import os, sys, glob, random, math, struct, zipfile, gc, time
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import SGD
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# ══════════════════════════════════════════════════════════════════════════════
#  THE ABLATION SELECTOR  —  set this per upload
# ══════════════════════════════════════════════════════════════════════════════
FINGERPRINT_MODE = "rates_only"   # one of:
#   "full"             A — all 9 channels (4 weight + 4 input-rate + 1 hidden-rate)
#   "rates_only"       B — weight channels neutralised; both rate groups kept
#   "weights_only"     C — rate channels neutralised; weight channels kept
#   "input_rate_only"  D — keep ONLY the 4 input-rate channels (pure encoder readout,
#                          STDP-independent); weights + hidden-rate neutralised
#   "hidden_rate_only" E — keep ONLY the 1 hidden-rate channel (network compute);
#                          weights + input-rate neutralised
assert FINGERPRINT_MODE in (
    "full", "rates_only", "weights_only", "input_rate_only", "hidden_rate_only")

# ══════════════════════════════════════════════════════════════════════════════
#  Fingerprints kept per recording session
# ══════════════════════════════════════════════════════════════════════════════
# The shards store 2 fingerprints per session. FPS_PER_SESSION=1 keeps exactly one
# (deterministic, seeded) for BOTH dev and test.
#
#   • test: session leakage becomes structurally impossible — no two test samples share
#     a recording, so every positive pair is cross-session by construction and the
#     "leaky" and "session-free" metrics coincide. Nothing left to mask.
#   • dev : halves samples/epoch (276,630 → ~138k) and halves epoch wall time.
#   • RAM : NOT halved. Weights are memmap'd, so residency is already ~0.
#
# Set to 2 to use every fingerprint (original behaviour).
FPS_PER_SESSION = 1
SEED            = 42

# ── Compute ───────────────────────────────────────────────────────────────────
USE_DATAPARALLEL = True        # use both T4s if present (auto-falls back to 1 GPU)
USE_AMP          = True        # fp16 autocast + GradScaler
NUM_WORKERS      = 4           # memmap reads are I/O-bound → more workers than before
PREFETCH         = 4

# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_DIR  = "/kaggle/input/datasets/qphulong/vox1and2-fingerprints-tripletstdp"
WORK_DIR   = "/kaggle/working"
DEV_GLOB   = os.path.join(INPUT_DIR, "vox2_*_fingerprints.npz")   # train (closed-set)
TEST_GLOB  = os.path.join(INPUT_DIR, "vox1_*_fingerprints.npz")   # eval  (open-set, disjoint)

MODE_TAG   = FINGERPRINT_MODE
BEST_CKPT  = os.path.join(WORK_DIR, f"best_model_{MODE_TAG}.pth")
LAST_CKPT  = os.path.join(WORK_DIR, f"last_state_{MODE_TAG}.pth")
RESUME     = True              # resume from LAST_CKPT if it exists

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU    = torch.cuda.device_count()
print("Device:", DEVICE, "| GPUs:", N_GPU)
if torch.cuda.is_available():
    for i in range(N_GPU):
        print(f"  GPU{i}:", torch.cuda.get_device_name(i))

# ── Input geometry  (compact → model input (9,33,384) at model time) ──────────
N_CHANNELS    = 96       # tonotopic auditory channels
N_PER_CHANNEL = 4        # input neurons / channel  (2 sustained, 1 onset, 1 phase)
N_H           = 384      # hidden neurons  = last axis of every tensor
R_OFF         = 16       # tonotopic offset radius → 33 offsets
IN_CH = 9                # 4 weight + 4 input-rate + 1 hidden-rate
H_OFF = 2 * R_OFF + 1    # 33
W_HID = N_H              # 384
NEUTRAL_FILL = math.exp(-30)   # zero-variance constant for ablated channels (≈9.4e-14)

# ── IN_IDX  (identical to _fingerprint_core.py) ───────────────────────────────
OFFSETS = np.arange(-R_OFF, R_OFF + 1)                               # (33,)
_ch_j   = np.arange(N_H) // N_PER_CHANNEL                            # (384,)
_in_ch  = (_ch_j[None, :] + OFFSETS[:, None]) % N_CHANNELS           # (33, 384)
IN_IDX  = np.stack([_in_ch * N_PER_CHANNEL + t for t in range(N_PER_CHANNEL)])  # (4,33,384)
IN_IDX_T = torch.as_tensor(IN_IDX, dtype=torch.long, device=DEVICE)

# ── Augmentation ──────────────────────────────────────────────────────────────
ROLL_MAX_CH = 2          # max tonotopic jitter (channels)

# ── Hyperparameters ───────────────────────────────────────────────────────────
EMBED_DIM    = 256
M            = 0.2       # ArcFace target margin (ramped in)
S            = 30        # ArcFace scale
LR           = 0.08      # peak LR (after warmup)
WEIGHT_DECAY = 5e-4
EPOCHS       = 30
WARMUP_EPOCHS        = 3
MARGIN_WARMUP_EPOCHS = 15
EARLY_STOP_PATIENCE  = 8
GRAD_CLIP            = 5.0
P_CLASSES    = 32        # classes per PK batch
K_SAMPLES    = 4         # samples per class → batch size = 128

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
print(f"\n{'='*60}\n  FINGERPRINT_MODE: {FINGERPRINT_MODE}\n{'='*60}")
print(f"Input {IN_CH}x{H_OFF}x{W_HID}  | DataParallel={USE_DATAPARALLEL and N_GPU>1} | AMP={USE_AMP}")


def rss_gb():
    """Anonymous+file RSS of this process, GB. Cheap sanity probe for the RAM fix."""
    try:
        with open("/proc/self/statm") as f:
            return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 1e9
    except Exception:
        return float("nan")


In [ ]:
from numpy.lib import format as npformat

# ══════════════════════════════════════════════════════════════════════════════
#  Zero-copy shard access
# ══════════════════════════════════════════════════════════════════════════════
# A .npz written by np.savez is a ZIP_STORED archive of verbatim .npy members. So the raw
# bytes of `weights` sit contiguously inside the file and can be np.memmap'd in place — no
# extraction, no decompression, no 28 GB of writable disk needed. Pages land in the OS page
# cache (reclaimable) rather than anonymous RAM (OOM-killable).
#
# If the shards were written with np.savez_COMPRESSED this cannot work: re-save them from
# prepare_fingerprints.ipynb with plain np.savez. Same file size (fp16 weights barely
# compress), and it makes this whole notebook fit in RAM.

def npz_member_offset(path, key):
    """Byte offset of the .npy stream for `key` inside an uncompressed .npz."""
    with zipfile.ZipFile(path) as zf:
        zi = zf.getinfo(key + ".npy")
        if zi.compress_type != zipfile.ZIP_STORED:
            raise ValueError(
                f"{os.path.basename(path)}:{key} is DEFLATED. Re-save the shards with "
                f"np.savez(...) instead of np.savez_compressed(...).")
        header_offset = zi.header_offset
    with open(path, "rb") as fh:
        fh.seek(header_offset)
        local = fh.read(30)
        nlen, elen = struct.unpack("<HH", local[26:30])
        return header_offset + 30 + nlen + elen


def npz_member_memmap(path, key, mode="r"):
    """np.memmap a member of an UNCOMPRESSED .npz. Returns a read-only ndarray view."""
    off = npz_member_offset(path, key)
    with open(path, "rb") as fh:
        fh.seek(off)
        ver = npformat.read_magic(fh)
        if ver == (1, 0):
            shape, fortran, dtype = npformat.read_array_header_1_0(fh)
        elif ver == (2, 0):
            shape, fortran, dtype = npformat.read_array_header_2_0(fh)
        else:                                            # numpy >= 2 private fallback
            shape, fortran, dtype = npformat._read_array_header(fh, ver)
        if fortran:
            raise ValueError("Fortran-ordered member; re-save C-contiguous.")
        data_off = fh.tell()
    return np.memmap(path, dtype=dtype, mode=mode, offset=data_off, shape=shape)


def read_small(path, *keys):
    """Read tiny members (labels / ids) without decompressing `weights`."""
    with np.load(path, allow_pickle=True) as d:
        return tuple(d[k] for k in keys)


# ── Shard discovery + metadata (no bulk data touched) ─────────────────────────
DEV_SHARDS  = sorted(glob.glob(DEV_GLOB))
TEST_SHARDS = sorted(glob.glob(TEST_GLOB))
if not DEV_SHARDS:  raise FileNotFoundError(f"no dev shards match: {DEV_GLOB}")
if not TEST_SHARDS: raise FileNotFoundError(f"no test shards match: {TEST_GLOB}")


def scan(paths):
    """(sizes, person_ids, record_ids) concatenated in `paths` order = global row order."""
    sizes, pids, recs = [], [], []
    for p in paths:
        pid, rec = read_small(p, "person_ids", "record_ids")
        sizes.append(len(pid)); pids.append(pid); recs.append(rec)
    return sizes, np.concatenate(pids), np.concatenate(recs)


DEV_SIZES,  pid_train_all, rec_train_all = scan(DEV_SHARDS)
TEST_SIZES, pid_val_all,   rec_val_all   = scan(TEST_SHARDS)


# ── One fingerprint per session ───────────────────────────────────────────────
def one_per_session(pids, recs, n_keep=1, seed=SEED):
    """Row indices keeping at most `n_keep` fingerprints from each (person, session).

    Selection is a seeded random draw within each session, not "take the first", so we
    don't systematically bias toward whichever utterance the preparer emitted first.
    Returned indices are sorted ascending → downstream row order is preserved, which keeps
    the (shard, row) index and the label vectors aligned.
    """
    key = np.char.add(np.char.add(np.asarray(pids, dtype=str), "|"),
                      np.asarray(recs, dtype=str))
    _, inv = np.unique(key, return_inverse=True)                 # session group id per row
    rnd = np.random.default_rng(seed).random(len(inv))
    order = np.lexsort((rnd, inv))                               # group-major, random within
    grp   = inv[order]
    rank  = np.arange(len(order)) - np.searchsorted(grp, grp)    # 0,1,2,... within group
    return np.sort(order[rank < n_keep])


def apply_subsample(pids, recs, split):
    if FPS_PER_SESSION is None:
        keep = np.arange(len(pids))
    else:
        keep = one_per_session(pids, recs, FPS_PER_SESSION)
    n_sess = len(np.unique(np.char.add(np.asarray(pids, dtype=str),
                                       np.asarray(recs, dtype=str))))
    print(f"{split:5s}: {len(pids):7d} → {len(keep):7d} samples "
          f"({len(keep)/len(pids):.1%})  over {n_sess} sessions "
          f"({len(keep)/n_sess:.2f} fp/session)")
    return keep


print(f"\nSubsampling to {FPS_PER_SESSION} fingerprint(s)/session:")
KEEP_DEV  = apply_subsample(pid_train_all, rec_train_all, "dev")
KEEP_TEST = apply_subsample(pid_val_all,   rec_val_all,   "test")

pid_train, rec_train = pid_train_all[KEEP_DEV],  rec_train_all[KEEP_DEV]
pid_val,   rec_val   = pid_val_all[KEEP_TEST],   rec_val_all[KEEP_TEST]
del pid_train_all, rec_train_all, pid_val_all, rec_val_all

# ── One-time sanity check: memmap view must equal the eager read ──────────────
_p  = DEV_SHARDS[0]
_mm = npz_member_memmap(_p, "weights")
_eager, = read_small(_p, "weights")
assert _mm.shape == _eager.shape and _mm.dtype == _eager.dtype, (_mm.shape, _eager.shape)
assert np.array_equal(np.asarray(_mm[:16]), _eager[:16]), "memmap offset is wrong!"
del _eager, _mm; gc.collect()
print("memmap OK — shards are ZIP_STORED, zero-copy access enabled")

_BYTES_PER_SAMPLE = N_PER_CHANNEL * H_OFF * W_HID * 2
print(f"\nDev  : {len(DEV_SHARDS):2d} shard(s), {len(KEEP_DEV):7d} used "
      f"of {sum(DEV_SIZES):7d} ({sum(DEV_SIZES)*_BYTES_PER_SAMPLE/1e9:5.2f} GB on disk, "
      f"0 GB resident)")
print(f"Test : {len(TEST_SHARDS):2d} shard(s), {len(KEEP_TEST):7d} used "
      f"of {sum(TEST_SIZES):7d} ({sum(TEST_SIZES)*_BYTES_PER_SAMPLE/1e9:5.2f} GB on disk, "
      f"0 GB resident)")

# ── Label spaces ──────────────────────────────────────────────────────────────
all_persons     = sorted(set(pid_train.tolist()))
NUM_CLASSES     = len(all_persons)
person_to_label = {p: i for i, p in enumerate(all_persons)}
y_train = np.array([person_to_label[p] for p in pid_train], dtype=np.int64)

val_persons  = sorted(set(pid_val.tolist()))
val_to_label = {p: i for i, p in enumerate(val_persons)}
y_val = np.array([val_to_label[p] for p in pid_val], dtype=np.int64)

# Session/recording ids for the test split → used to EXCLUDE same-session pairs at eval.
rec_val_int = np.unique(rec_val, return_inverse=True)[1].astype(np.int64)

# With 1 fp/session every recording is unique → the same-session mask degenerates to the
# self-match mask, and the leaky / session-free metrics are identical by construction.
HAS_DUP_SESSIONS = len(rec_val_int) != len(np.unique(rec_val_int))

_bc = np.bincount(y_train)
print(f"\nTrain : {len(y_train)} samples  classes={NUM_CLASSES}  "
      f"per-class min/max={_bc.min()}/{_bc.max()}")
print(f"Val   : {len(y_val)} samples  classes={len(val_persons)} (disjoint, open-set)  "
      f"sessions={rec_val_int.max()+1}")
assert not (set(all_persons) & set(val_persons)), "train/test speakers overlap!"

if _bc.min() < K_SAMPLES:
    n_thin = int((_bc < K_SAMPLES).sum())
    print(f"! {n_thin} dev class(es) now have < K_SAMPLES={K_SAMPLES} fingerprints; "
          f"PKBatchSampler will sample them WITH replacement.")
if not HAS_DUP_SESSIONS:
    print("Test sessions are unique → session leakage is structurally impossible; "
          "leaky == session-free.")
print(f"RSS after metadata scan: {rss_gb():.2f} GB")


In [ ]:
class FingerprintAugment:
    """Online augmentation on the COMPACT triple (per sample, CPU). Tonotopically
    consistent: a roll by 4*delta on the 384 axis shifts whole auditory channels across
    weights / input_activity / hidden_activity alike (the IN_IDX gather is relative, so
    the expanded input-rate shifts with it)."""
    def __call__(self, w, ia, ha):
        # Mild circular tonotopic jitter (small shift → mic-response invariance, not a
        # full vocal-tract-length change that would destroy speaker identity).
        if ROLL_MAX_CH > 0 and random.random() < 0.5:
            shift = N_PER_CHANNEL * random.randint(-ROLL_MAX_CH, ROLL_MAX_CH)
            if shift:
                w  = torch.roll(w,  shifts=shift, dims=-1)
                ia = torch.roll(ia, shifts=shift, dims=-1)
                ha = torch.roll(ha, shifts=shift, dims=-1)

        # Additive Gaussian noise (p=0.5), per-array scale.
        if random.random() < 0.5:
            w  = w  + torch.randn_like(w)  * (0.01 * w.std().clamp(min=1e-6))
            ia = ia + torch.randn_like(ia) * (0.01 * ia.std().clamp(min=1e-6))
            ha = ha + torch.randn_like(ha) * (0.01 * ha.std().clamp(min=1e-6))

        # Global amplitude scale (p=0.5).
        if random.random() < 0.5:
            s = random.uniform(0.95, 1.05)
            w, ia, ha = w * s, ia * s, ha * s

        return w, ia, ha


In [ ]:
class ShardedFingerprintDataset(Dataset):
    """Global (shard, row) index over N shards; memmaps opened LAZILY, per worker process.

    Opening them in __init__ would make the parent hold N file mappings and would hand
    every forked worker a stale mapping object. Instead each worker builds its own cache on
    first touch, so the parent's RSS stays flat and `persistent_workers=True` keeps them warm.
    """
    def __init__(self, paths, sizes, y, augment=False, keep=None):
        self.paths = list(paths)
        index = np.empty((sum(sizes), 2), dtype=np.int64)
        o = 0
        for s, n in enumerate(sizes):
            index[o:o+n, 0] = s
            index[o:o+n, 1] = np.arange(n)
            o += n
        # `keep` selects a subset of the global row order (e.g. 1 fingerprint/session).
        # It is applied here, so shards on disk are untouched and the labels handed in
        # must already be the subset-aligned ones.
        self.index = index if keep is None else index[np.asarray(keep, dtype=np.int64)]
        assert len(self.index) == len(y), (len(self.index), len(y))
        self.y   = torch.from_numpy(np.ascontiguousarray(y)).long()
        self.aug = FingerprintAugment() if augment else None
        self._mm = None                                   # per-process memmap cache

    def _maps(self, s):
        if self._mm is None:
            self._mm = {}
        m = self._mm.get(s)
        if m is None:
            p = self.paths[s]
            m = (npz_member_memmap(p, "weights"),
                 npz_member_memmap(p, "input_activity"),
                 npz_member_memmap(p, "hidden_activity"))
            self._mm[s] = m
        return m

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        s, r = int(self.index[i, 0]), int(self.index[i, 1])
        W, IA, HA = self._maps(s)
        # np.array(...) forces a real copy out of the mapping before the tensor is made.
        w  = torch.from_numpy(np.array(W[r],  dtype=np.float16)).float()
        ia = torch.from_numpy(np.array(IA[r], dtype=np.float16)).float()
        ha = torch.from_numpy(np.array(HA[r], dtype=np.float16)).float()
        if self.aug is not None:
            w, ia, ha = self.aug(w, ia, ha)
        return w, ia, ha, self.y[i]


def expand_batch(w, ia, ha, mode):
    """Compact batch (on DEVICE) → model input (B,9,33,384). Ablated channel groups are
    replaced by a zero-variance constant (neutralised by the input BatchNorm), so the
    architecture / param-count is identical across all modes — only the visible input
    information changes.   w:(B,4,33,384)  ia,ha:(B,384)."""
    B = w.shape[0]
    in_rate  = ia[:, IN_IDX_T]                                   # (B,4,33,384)
    hid_rate = ha[:, None, None, :].expand(B, 1, H_OFF, W_HID)   # (B,1,33,384)

    drop_w  = mode in ("rates_only", "input_rate_only", "hidden_rate_only")
    drop_ir = mode in ("weights_only", "hidden_rate_only")
    drop_hr = mode in ("weights_only", "input_rate_only")
    if drop_w:  w        = torch.full_like(w,        NEUTRAL_FILL)
    if drop_ir: in_rate  = torch.full_like(in_rate,  NEUTRAL_FILL)
    if drop_hr: hid_rate = torch.full_like(hid_rate, NEUTRAL_FILL)
    return torch.cat([w, in_rate, hid_rate], dim=1)              # (B,9,33,384)


class PKBatchSampler:
    """Yields batches with exactly K samples from each of P randomly chosen classes.
    Now drawn from the FULL dev class set (6k+), not just whichever shard-group was
    resident — the shard-group restriction was starving the AAM head."""
    def __init__(self, labels, P, K):
        self.K = K
        arr = np.asarray(labels)
        self.cls_idx = defaultdict(list)
        for i, l in enumerate(arr):
            self.cls_idx[int(l)].append(i)
        self.cls_idx = {c: np.asarray(v) for c, v in self.cls_idx.items()}
        self.classes   = np.asarray(sorted(self.cls_idx.keys()))
        self.P         = min(P, len(self.classes))
        self.n_batches = max(1, len(arr) // (self.P * K))

    def __iter__(self):
        for _ in range(self.n_batches):
            batch = []
            for c in np.random.choice(self.classes, self.P, replace=False):
                idx = self.cls_idx[int(c)]
                batch.extend(np.random.choice(idx, self.K,
                                              replace=(len(idx) < self.K)).tolist())
            yield batch

    def __len__(self):
        return self.n_batches


_pin = torch.cuda.is_available()

train_ds = ShardedFingerprintDataset(DEV_SHARDS,  DEV_SIZES,  y_train,
                                     augment=True,  keep=KEEP_DEV)
val_ds   = ShardedFingerprintDataset(TEST_SHARDS, TEST_SIZES, y_val,
                                     augment=False, keep=KEEP_TEST)

train_sampler = PKBatchSampler(y_train, P_CLASSES, K_SAMPLES)
train_loader = DataLoader(
    train_ds, batch_sampler=train_sampler,
    num_workers=NUM_WORKERS, pin_memory=_pin,
    prefetch_factor=(PREFETCH if NUM_WORKERS > 0 else None),
    persistent_workers=(NUM_WORKERS > 0))

val_loader = DataLoader(
    val_ds, batch_size=64, shuffle=False,
    num_workers=max(2, NUM_WORKERS // 2), pin_memory=_pin,
    persistent_workers=True)

print(f"Train batches/epoch = {len(train_sampler)}  (batch = {P_CLASSES * K_SAMPLES})")
print(f"Val samples         = {len(val_ds)}")
print(f"Bytes read/epoch    ≈ {len(train_sampler)*P_CLASSES*K_SAMPLES*_BYTES_PER_SAMPLE/1e9:.1f} GB "
      f"(page cache, reclaimable)")
print(f"RSS after loader construction: {rss_gb():.2f} GB")


In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=(1, 1)):
        super().__init__()
        s = stride if isinstance(stride, tuple) else (stride, stride)
        self.conv1 = nn.Conv2d(in_ch,  out_ch, 3, stride=s, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch); self.act1 = nn.PReLU(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch); self.act2 = nn.PReLU(out_ch)
        self.shortcut = (
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=s, bias=False),
                          nn.BatchNorm2d(out_ch))
            if in_ch != out_ch or s != (1, 1) else nn.Identity())

    def forward(self, x):
        out = self.act1(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.act2(out + self.shortcut(x))


class AttentiveStatsPool(nn.Module):
    """Attentive mean+std pooling over the flattened spatial map. (B,C,L) → (B,2C)."""
    def __init__(self, in_dim, att_dim=128):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Conv1d(in_dim, att_dim, 1), nn.Tanh(),
            nn.Conv1d(att_dim, in_dim, 1))

    def forward(self, x):                                   # (B,C,L)
        w     = torch.softmax(self.attention(x), dim=2)
        mu    = (w * x).sum(dim=2)
        sigma = torch.sqrt((w * x.pow(2)).sum(dim=2).sub(mu.pow(2)).clamp(min=1e-4))
        return torch.cat([mu, sigma], dim=1)                # (B,2C)


class FingerprintCNN(nn.Module):
    """(B,9,33,384) → (B,embed_dim). Input BatchNorm standardises the 9 heterogeneous
    channels (weights ~[0,0.12] vs rates ~[0,1]); 2D ResNet downsamples the 384 hidden
    axis (W) hard, the 33 offset axis (H) gently; attentive stats → FC embedding."""
    def __init__(self, in_ch=9, embed_dim=256):
        super().__init__()
        self.in_bn = nn.BatchNorm2d(in_ch)
        self.stem  = nn.Sequential(
            nn.Conv2d(in_ch, 32, (3, 7), stride=1, padding=(1, 3), bias=False),
            nn.BatchNorm2d(32), nn.PReLU(32))                  # (B,32,33,384)
        self.block1 = ResBlock(32,  64,  stride=(1, 2))        # (B, 64,33,192)
        self.block2 = ResBlock(64,  128, stride=(2, 2))        # (B,128,17, 96)
        self.block3 = ResBlock(128, 256, stride=(2, 2))        # (B,256, 9, 48)
        self.block4 = ResBlock(256, 256, stride=(2, 2))        # (B,256, 5, 24)
        self.pool    = AttentiveStatsPool(256)                 # (B,512)
        self.dropout = nn.Dropout(p=0.4)
        self.fc      = nn.Linear(512, embed_dim, bias=False)
        self.bn      = nn.BatchNorm1d(embed_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)

    def forward(self, x):
        x = self.in_bn(x)
        x = self.stem(x)
        x = self.block1(x); x = self.block2(x); x = self.block3(x); x = self.block4(x)
        x = x.flatten(2)            # (B,256,H*W)
        x = self.pool(x)            # (B,512)
        x = self.dropout(x)
        x = self.bn(self.fc(x))
        return x                    # (B,embed_dim) — BN-normalised, L2 at eval time


class AAMSoftmax(nn.Module):
    """Additive Angular Margin Softmax (ArcFace), with the easy-margin guard.
    The margin is settable (set_margin) so it can be warmed up 0→M during training."""
    def __init__(self, embed_dim, num_classes, m=0.2, s=30):
        super().__init__()
        self.s = s
        self.W = nn.Parameter(torch.empty(num_classes, embed_dim))
        nn.init.xavier_normal_(self.W)
        self.set_margin(m)

    def set_margin(self, m):
        self.m = m
        self.cos_m, self.sin_m = math.cos(m), math.sin(m)
        self.th, self.mm = math.cos(math.pi - m), math.sin(math.pi - m) * m

    def forward(self, embeddings, labels):
        cos = F.linear(F.normalize(embeddings), F.normalize(self.W)).clamp(-1 + 1e-7, 1 - 1e-7)
        sin = torch.sqrt((1.0 - cos.pow(2)).clamp(min=1e-7))
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)
        oh  = F.one_hot(labels, num_classes=self.W.size(0)).float()
        logits = (oh * phi + (1.0 - oh) * cos) * self.s
        return F.cross_entropy(logits, labels)


model = FingerprintCNN(IN_CH, embed_dim=EMBED_DIM).to(DEVICE)
aam   = AAMSoftmax(EMBED_DIM, NUM_CLASSES, m=M, s=S).to(DEVICE)
if USE_DATAPARALLEL and N_GPU > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel over {N_GPU} GPUs")

_core = model.module if isinstance(model, nn.DataParallel) else model
total = sum(p.numel() for p in _core.parameters()) + sum(p.numel() for p in aam.parameters())
print(f"Total params: {total:,}  (~{total/1e6:.2f}M)")
with torch.no_grad():
    dummy = torch.randn(4, IN_CH, H_OFF, W_HID, device=DEVICE)
    print("Embedding shape:", model(dummy).shape)   # expect (4, EMBED_DIM)
del dummy


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Open-set verification metrics — SESSION-LEAKAGE AWARE
# ══════════════════════════════════════════════════════════════════════════════
# Every test embedding is both a query and a gallery item. Two exclusions:
#   1. self-match                     (always)
#   2. same-recording / same-session  (when rec_ids is given)
#
# (2) matters a great deal here. Each session contributes exactly 2 fingerprints, so every
# query has exactly ONE same-session positive — same mic, same channel, same room. Cosine
# retrieves it almost for free. Leaving it in makes R@1 and EER report a blend of speaker
# verification and session re-identification, and the session part dominates Rank-1.
#
# Pass rec_ids=None to reproduce the old (leaky) numbers for comparison.

@torch.no_grad()
def encode_all(loader, mdl):
    mdl.eval()
    embs, lbls = [], []
    for w, ia, ha, y in loader:
        x = expand_batch(w.to(DEVICE, non_blocking=True),
                         ia.to(DEVICE, non_blocking=True),
                         ha.to(DEVICE, non_blocking=True), FINGERPRINT_MODE)
        e = F.normalize(mdl(x), dim=1)
        embs.append(e.float().cpu()); lbls.append(y)
    return torch.cat(embs), torch.cat(lbls)


@torch.no_grad()
def eval_metrics(emb, lbl, rec_ids=None, block=1024, n_bins=4000):
    """Memory-bounded open-set metrics → (Rank-1, Rank-5, EER, mAP).

    Sweeps row-BLOCKS (peak ≈ block×N) instead of materialising the full N×N cosine matrix.
    Dropped pairs get score -2.0 so they sort strictly below every real candidate, which
    keeps the average-precision ranks correct without a per-row compaction.
    """
    dev = DEVICE if torch.cuda.is_available() else torch.device("cpu")
    E, L = emb.to(dev), lbl.to(dev)
    N = E.size(0)
    R = torch.as_tensor(rec_ids, device=dev) if rec_ids is not None else None

    all_cols = torch.arange(N, device=dev)
    pos_h = torch.zeros(n_bins, device=dev)          # same-speaker score histogram
    neg_h = torch.zeros(n_bins, device=dev)          # diff-speaker score histogram
    r1_hits = r5_hits = n_query = 0
    ap_sum, ap_cnt = 0.0, 0
    k_top = min(5, N - 1)
    pos_ranks = torch.arange(1, N + 1, device=dev).float()

    for r0 in range(0, N, block):
        r1e   = min(r0 + block, N)
        rows  = torch.arange(r0, r1e, device=dev)
        local = torch.arange(r1e - r0, device=dev)

        S    = E[r0:r1e] @ E.t()                      # (b,N) cosine sims
        same = L[r0:r1e, None] == L[None, :]

        drop = torch.zeros_like(same)
        drop[local, rows] = True                      # self-match
        if R is not None:
            drop |= (R[r0:r1e, None] == R[None, :])   # same session

        S    = S.masked_fill(drop, -2.0)
        same = same & ~drop

        nrel  = same.sum(1)
        valid = nrel > 0                              # queries with ≥1 usable positive

        # Rank-1 / Rank-5 (over valid queries only)
        top = S.topk(k_top, dim=1).indices
        hit = same.gather(1, top)
        r1_hits += hit[valid, 0].sum().item()
        r5_hits += hit[valid].any(1).sum().item()
        n_query += int(valid.sum().item())

        # mAP (per-row average precision)
        order = S.argsort(dim=1, descending=True)
        rel   = same.gather(1, order).float()
        cum   = rel.cumsum(1)
        ap    = ((cum / pos_ranks) * rel).sum(1) / nrel.clamp(min=1)
        ap_sum += ap[valid].sum().item(); ap_cnt += int(valid.sum().item())

        # EER histograms — upper triangle only (j > i), dropped pairs excluded entirely
        keep  = (all_cols[None, :] > rows[:, None]) & ~drop
        sflat, smask = S[keep], same[keep]
        pos_h += torch.histc(sflat[smask],  bins=n_bins, min=-1.0, max=1.0)
        neg_h += torch.histc(sflat[~smask], bins=n_bins, min=-1.0, max=1.0)

        del S, same, drop, order, rel, cum, top, hit, keep, sflat, smask

    r1  = r1_hits / max(n_query, 1)
    r5  = r5_hits / max(n_query, 1)
    mAP = ap_sum / max(ap_cnt, 1)

    # EER: threshold at each bin edge; far(t)=neg≥t / Nneg, frr(t)=pos<t / Npos.
    P, Ng  = pos_h.sum(), neg_h.sum()
    neg_ge = torch.flip(torch.cumsum(torch.flip(neg_h, [0]), 0), [0])
    pos_lt = torch.cumsum(pos_h, 0) - pos_h
    far, frr = neg_ge / Ng.clamp(min=1), pos_lt / P.clamp(min=1)
    j   = torch.argmin((far - frr).abs())
    eer = float(((far[j] + frr[j]) / 2).item())

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return r1, r5, eer, mAP


def evaluate(mdl):
    """Returns (clean, leaky) metric tuples. `clean` excludes same-session pairs.

    At 1 fingerprint/session every recording is unique, so the same-session mask collapses
    onto the self-match mask and the two are provably identical — we skip the second pass
    rather than pay ~30 s/epoch to recompute the same numbers.
    """
    emb, lbl = encode_all(val_loader, mdl)
    clean = eval_metrics(emb, lbl, rec_ids=rec_val_int)
    leaky = eval_metrics(emb, lbl, rec_ids=None) if HAS_DUP_SESSIONS else clean
    del emb, lbl; gc.collect()
    return clean, leaky


print("Eval utilities loaded — session-leakage-aware, memory-bounded, block-wise.")


In [ ]:
def build_optimizer(mdl, aam_head, lr, wd):
    """Weight-decay only on conv/linear weights, not BN/PReLU/biases."""
    no_wd_types = (nn.BatchNorm1d, nn.BatchNorm2d, nn.PReLU)
    decay, no_decay = [], []
    for module in list(mdl.modules()) + list(aam_head.modules()):
        if isinstance(module, no_wd_types):
            no_decay.extend(module.parameters(recurse=False))
        elif hasattr(module, "weight") and module.weight is not None:
            if module.weight.requires_grad: decay.append(module.weight)
        if hasattr(module, "bias") and module.bias is not None:
            if module.bias.requires_grad: no_decay.append(module.bias)
    decay.append(aam_head.W)        # nn.Parameter, not a module weight
    seen = set()
    decay    = [p for p in decay    if id(p) not in seen and not seen.add(id(p))]
    no_decay = [p for p in no_decay if id(p) not in seen and not seen.add(id(p))]
    return SGD([{"params": decay, "weight_decay": wd},
                {"params": no_decay, "weight_decay": 0.0}],
               lr=lr, momentum=0.9, nesterov=True)


optimizer = build_optimizer(model, aam, LR, WEIGHT_DECAY)
warmup    = LinearLR(optimizer, start_factor=0.1, total_iters=max(1, WARMUP_EPOCHS))
cosine    = CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS - WARMUP_EPOCHS), eta_min=1e-5)
scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[max(1, WARMUP_EPOCHS)])
scaler    = torch.amp.GradScaler("cuda", enabled=USE_AMP and torch.cuda.is_available())

HIST_KEYS = ["loss", "rank1", "rank5", "eer", "map",          # session-free (reported)
             "rank1_leak", "rank5_leak", "eer_leak", "map_leak"]  # legacy, for comparison
history  = {k: [] for k in HIST_KEYS}
best_eer, best_epoch, epochs_no_improve, start_epoch = float("inf"), 0, 0, 1


def _core_of(m):
    return m.module if isinstance(m, nn.DataParallel) else m


def save_last(epoch):
    """Full training state — a dead kernel costs one epoch, not the whole run."""
    torch.save({"model": _core_of(model).state_dict(), "aam": aam.state_dict(),
                "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "history": history, "epoch": epoch,
                "best_eer": best_eer, "best_epoch": best_epoch,
                "epochs_no_improve": epochs_no_improve, "mode": FINGERPRINT_MODE},
               LAST_CKPT)


# ── Resume ────────────────────────────────────────────────────────────────────
if RESUME and os.path.exists(LAST_CKPT):
    _ck = torch.load(LAST_CKPT, map_location=DEVICE, weights_only=False)
    if _ck.get("mode") != FINGERPRINT_MODE:
        print(f"! {LAST_CKPT} is mode={_ck.get('mode')}, not {FINGERPRINT_MODE} — ignoring.")
    else:
        _core_of(model).load_state_dict(_ck["model"]); aam.load_state_dict(_ck["aam"])
        optimizer.load_state_dict(_ck["optimizer"]); scheduler.load_state_dict(_ck["scheduler"])
        scaler.load_state_dict(_ck["scaler"])
        history = _ck["history"]
        for k in HIST_KEYS:
            history.setdefault(k, [])
        best_eer, best_epoch = _ck["best_eer"], _ck["best_epoch"]
        epochs_no_improve = _ck["epochs_no_improve"]
        start_epoch = _ck["epoch"] + 1
        print(f"Resumed from ep{_ck['epoch']} (best EER={best_eer:.3f} @ ep{best_epoch})")
    del _ck


for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()

    # ── Margin warmup: ramp ArcFace m 0→M ─────────────────────────────────────
    cur_m = M * min(1.0, epoch / MARGIN_WARMUP_EPOCHS)
    aam.set_margin(cur_m)
    model.train(); aam.train()

    total_loss, n_batches = 0.0, 0
    for w, ia, ha, y in train_loader:
        w  = w.to(DEVICE, non_blocking=True);  ia = ia.to(DEVICE, non_blocking=True)
        ha = ha.to(DEVICE, non_blocking=True); y  = y.to(DEVICE, non_blocking=True)
        x = expand_batch(w, ia, ha, FINGERPRINT_MODE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16,
                            enabled=USE_AMP and torch.cuda.is_available()):
            emb = model(x)
        loss = aam(emb.float(), y)                # ArcFace math in fp32 (autocast off)
        scaler.scale(loss).backward()
        if GRAD_CLIP > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                list(model.parameters()) + list(aam.parameters()), GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item(); n_batches += 1

    scheduler.step()
    avg_loss = total_loss / max(n_batches, 1)

    # ── Eval (open-set, session-free + legacy leaky) ───────────────────────────
    (r1, r5, eer, mAP), (r1L, r5L, eerL, mAPL) = evaluate(model)
    for k, v in zip(HIST_KEYS,
                    (avg_loss, r1, r5, eer, mAP, r1L, r5L, eerL, mAPL)):
        history[k].append(v)

    # ── Checkpoint on best SESSION-FREE EER + early-stop bookkeeping ───────────
    marker = ""
    if eer < best_eer:
        best_eer, best_epoch, epochs_no_improve = eer, epoch, 0
        torch.save({"model": _core_of(model).state_dict(), "aam": aam.state_dict(),
                    "epoch": epoch, "rank1": r1, "eer": eer,
                    "rank1_leak": r1L, "eer_leak": eerL,
                    "mode": FINGERPRINT_MODE}, BEST_CKPT)
        marker = " ★"
    else:
        epochs_no_improve += 1
    save_last(epoch)

    print(f"Ep {epoch:3d}/{EPOCHS}  m={cur_m:.2f}  loss={avg_loss:.4f}  "
          f"R@1={r1:.3f} R@5={r5:.3f} EER={eer:.3f} mAP={mAP:.3f}  |  "
          f"leaky R@1={r1L:.3f} EER={eerL:.3f}  |  "
          f"{time.time()-t0:5.0f}s  RSS={rss_gb():.1f}GB{marker}")

    # ── Early stop ────────────────────────────────────────────────────────────
    if EARLY_STOP_PATIENCE > 0 and epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stop at epoch {epoch} — session-free EER no improvement for "
              f"{EARLY_STOP_PATIENCE} epochs (best @ ep{best_epoch}).")
        break

print(f"\n[{MODE_TAG}] Best session-free EER: {best_eer:.3f} @ ep{best_epoch}  → {BEST_CKPT}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))

axes[0].plot(history["loss"], color="steelblue")
axes[0].set(xlabel="Epoch", ylabel="Loss", title=f"AAM-Softmax Loss [{MODE_TAG}]")
axes[0].grid(alpha=0.3)

axes[1].plot(history["rank1"], label="Rank-1", color="steelblue")
axes[1].plot(history["rank5"], label="Rank-5", color="darkorange")
axes[1].plot(history["eer"],   label="EER",    color="firebrick", linestyle="--")
axes[1].set(xlabel="Epoch", ylabel="Score",
            title=f"Session-free retrieval [{MODE_TAG}]", ylim=(0, 1))
axes[1].legend(); axes[1].grid(alpha=0.3)

# The session-leakage gap — how much of the old numbers was session re-identification.
axes[2].plot(history["rank1"],      label="R@1 (session-free)", color="steelblue")
axes[2].plot(history["rank1_leak"], label="R@1 (leaky)", color="steelblue", linestyle=":")
axes[2].plot(history["eer"],        label="EER (session-free)", color="firebrick")
axes[2].plot(history["eer_leak"],   label="EER (leaky)", color="firebrick", linestyle=":")
axes[2].set(xlabel="Epoch", ylabel="Score", title="Session-leakage gap", ylim=(0, 1))
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, f"training_curves_{MODE_TAG}.png"), dpi=120)
plt.show()

b = int(np.argmin(history["eer"]))
print(f"[{MODE_TAG}] Best epoch {b+1}")
print(f"  session-free : R@1={history['rank1'][b]:.3f}  R@5={history['rank5'][b]:.3f}  "
      f"EER={history['eer'][b]:.3f}  mAP={history['map'][b]:.3f}")
print(f"  leaky (old)  : R@1={history['rank1_leak'][b]:.3f}  R@5={history['rank5_leak'][b]:.3f}  "
      f"EER={history['eer_leak'][b]:.3f}  mAP={history['map_leak'][b]:.3f}")
print(f"  Δ from session leakage: R@1 {history['rank1_leak'][b]-history['rank1'][b]:+.3f}   "
      f"EER {history['eer_leak'][b]-history['eer'][b]:+.3f}")


In [ ]:
# t-SNE of the test embeddings. 41k points / 1251 speakers is unreadable, so a fixed
# random subset of speakers is shown (all of their samples), coloured by speaker.
import matplotlib as mpl

N_TSNE_SPEAKERS = 15

ckpt = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
_core_of(model).load_state_dict(ckpt["model"])

emb, lbl = encode_all(val_loader, model)               # val_loader is shuffle=False
assert np.array_equal(lbl.numpy(), y_val), "embedding order misaligned with labels"
emb_np  = emb.numpy()
pid_arr = np.asarray(pid_val)

rng   = np.random.default_rng(42)
shown = rng.choice(np.asarray(val_persons), size=min(N_TSNE_SPEAKERS, len(val_persons)),
                   replace=False)
sel   = np.isin(pid_arr, shown)
sub, sub_pid = emb_np[sel], pid_arr[sel]

perp   = min(30, len(sub) - 1)
coords = TSNE(n_components=2, perplexity=perp, init="pca", random_state=42).fit_transform(sub)

cmap = mpl.colormaps["tab20"].resampled(len(shown))
fig, ax = plt.subplots(figsize=(9, 8))
for i, person in enumerate(sorted(shown)):
    m = sub_pid == person
    ax.scatter(coords[m, 0], coords[m, 1], color=cmap(i), s=28,
               edgecolors="black", linewidths=0.3, alpha=0.85, label=person)
ax.set(xlabel="t-SNE 1", ylabel="t-SNE 2",
       title=f"Test embeddings [{MODE_TAG}] — {len(shown)} of {len(val_persons)} "
             f"disjoint speakers\nsession-free EER={ckpt.get('eer', float('nan')):.3f}  "
             f"R@1={ckpt.get('rank1', float('nan')):.3f}")
ax.legend(ncol=2, fontsize=7, markerscale=1.2, loc="best", framealpha=0.9)
plt.tight_layout()
out_path = os.path.join(WORK_DIR, f"tsne_test_{MODE_TAG}.png")
plt.savefig(out_path, dpi=120); plt.show()
print(f"Saved → {out_path}")
del emb, lbl, emb_np; gc.collect()
